In [14]:

import pandas as pd
import torch 
from transformers import AutoTokenizer, AutoModelForCausalLM

In [15]:
import os 
access_token = os.environ.get('HF_TOKEN')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [16]:
!nvidia-smi

Fri Feb  6 15:20:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             62W /  400W |   16283MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [17]:
model_id = "Qwen/Qwen3-8B"

tokenizer  = AutoTokenizer.from_pretrained(model_id,
                                           trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(model_id,
                                             device_map="auto",
                                             torch_dtype="auto",
                                             trust_remote_code=True
)

model.eval()




Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
        (post_attention_la

In [ ]:
import torch

SYSTEM = f"""You are an expert writer of short argumentative texts written in clear, natural academic prose.

Produce a concise argumentative microtext that reads like a brief excerpt from an academic essay, while fully respecting all structural requirements given in the task.

Write only the microtext itself.
Do not explain your reasoning.
Do not include meta-comments or additional text.
"""
# f"""You are an expert writer of short argumentative texts written in clear, natural academic prose.

# Your task is to produce well-formed argumentative microtexts that meet all specified structural requirements.

# Write only the microtext itself.
# Do not explain your reasoning.
# Do not include meta-comments or additional text.





# f"""You are an expert writer of argumentative microtexts for academic argumentation analysis.

# You must strictly follow all structural constraints.
# Do not explain your reasoning.
# Do not add meta-comments.
# Output only the microtext in the required format.
#           """
          
# "You are a writing assistant. Follow the user's rules exactly. "
# "Output ONLY the requested microtext paragraph. "
# "Do not include analysis, explanations, or meta-commentary."


def build_user_prompt(topic: str, stance: str) -> str:
    return f"""Write a short argumentative microtext on the topic below.

Topic: {topic}  
Stance: {stance}

The text should be written as a single coherent paragraph consisting of 4 to 6 sentences.

One sentence should clearly state the main claim and express the author’s stance.
The remaining sentences should develop the argument by giving reasons, addressing a possible objection, and responding to that objection.

Include at least one sentence that presents an opposing viewpoint, and at least one sentence that responds to it.

Do not repeat or reformulate the main claim once it has been stated.
The final sentence should introduce a new argumentative reason rather than summarizing the argument.

Each sentence should express a complete argumentative thought, and the argument should be understandable on its own.

Write in clear, natural academic prose.
Do not include explanations, labels, or text outside the argument itself.
Avoid explicit conclusion markers such as “therefore”, “thus”, “in conclusion”, or “ultimately”.

"""








# f"""
# Write an argumentative microtext that follows the rules below.

# Rules:
# 1. The text must consist of 4 to 6 sentences.
# 2. Exactly one sentence must express the main claim and clearly state the author’s stance.
# 3. All other sentences must function as argumentative reasons that either support the main claim or challenge it.
# 4. At least one sentence must present a clear opposing viewpoint.
# 5. If an opposing viewpoint is presented, at least one sentence must respond to or weaken it.
# 6. The final sentence must present a new argumentative reason and must not summarize or restate the main claim.
# 7. Each sentence must express a complete argumentative proposition.
# 8. The argument must be understandable on its own without relying on unstated assumptions.

# Topic: {topic}
# Stance: {stance}

# Write the microtext as a single coherent paragraph.
# Do not include explanations, labels, or any additional text.
# Do not use conclusion markers such as “therefore”, “thus”, “in conclusion”, or “ultimately”.


# """

# prompt_B_re = f"""Write an argumentative microtext that follows the rules below.

# Rules:
# 1. The text must consist of 4 to 6 sentences.
# 2. Exactly one sentence must express the main claim and clearly state the author’s stance.
# 3. All other sentences must function as argumentative reasons that either support the main claim or challenge it.
# 4. At least one sentence must present a clear opposing viewpoint.
# 5. If an opposing viewpoint is presented, at least one sentence must respond to or weaken it.
# 6. The text must end with an argumentative reason (support or response), not with a summary or conclusion.
# 7. Do not restate or reformulate the main claim after it is stated.
# 8. Each sentence must express a complete argumentative proposition.
# 9. The argument must be understandable on its own without relying on unstated assumptions.

# Topic: {topic}
# Stance: {stance}

# Write the microtext as a single coherent paragraph.
# Do not include explanations, labels, or any additional text.
# """

# prompt_A_re = f"""Write an argumentative microtext that follows the rules below.

# Rules:
# 1. The text must consist of exactly 4 sentences.
# 2. Exactly one sentence must express the main claim and clearly state the author’s stance.
# 3. The other sentences must function as argumentative reasons that either support the main claim or challenge it.
# 4. At least one sentence must present a clear opposing viewpoint.
# 5. If an opposing viewpoint is presented, one sentence must respond to or weaken it.
# 6. Do not restate or reformulate the main claim after it is stated.
# 7. Do not include neutral summaries or balanced conclusions.
# 8. Each sentence must express a complete argumentative proposition.

# Topic: {topic}
# Stance: {stance}

# Write the microtext as a single coherent paragraph.
# Do not include explanations, labels, or any additional text.
# """


# prompt_B = f"""Write an argumentative microtext that follows the rules below.

# Rules:
# 1. The text must consist of 4 to 6 sentences.
# 2. Exactly one sentence must express the main claim and clearly state the author’s stance.
# 3. All other sentences must function as reasons that either support the main claim or challenge it.
# 4. At least one sentence must present a clear opposing viewpoint.
# 5. If an opposing viewpoint is presented, at least one sentence must respond to or weaken it.
# 6. Do not restate, reformulate, or summarize the main claim after it is stated.
# 7. Do not include any concluding, synthesizing, or evaluative sentence.
# 8. Each sentence must express a complete argumentative proposition.
# 9. The argument must be understandable on its own without relying on unstated assumptions.

# Topic: {topic}
# Stance: {stance}

# Write the microtext as a single coherent paragraph.
# Do not include explanations, labels, or any additional text.

#    prompt_A = f"""Write an argumentative microtext that follows the rules below.

# Rules:
# 1. The text must consist of exactly 4 sentences.
# 2. Sentence 1 must state the single main claim and express the author’s stance.
# 3. Sentence 2 must provide a reason that supports the main claim.
# 4. Sentence 3 must present a clear counter-argument against the main claim.
# 5. Sentence 4 must respond to or weaken the counter-argument.
# 6. Do not restate, reformulate, or summarize the main claim after sentence 1.
# 7. Do not include any concluding, synthesizing, or evaluative sentence.
# 8. Each sentence must express a complete argumentative proposition.
# 9. The argument must be understandable on its own without relying on unstated assumptions.

# Topic: {topic}
# Stance: {stance}

# Write the microtext as a single coherent paragraph.
# Do not include explanations, labels, or any additional text.

# """
#     return f"""Write a short argumentative microtext that follows the rules below.

# Rules:
# 1. The text must consist of 4 to 6 sentences.
# 2. Each sentence must express a complete argumentative proposition.
# 3. The text must contain exactly one clear main claim that expresses the author’s stance.
# 4. The remaining sentences must provide reasons that either support the main claim or present a clear counter-argument.
# 5. At least one sentence must express an opposing viewpoint.
# 6. The author must take a clear stance; do not write a neutral or balanced overview.
# 7. Do not include background information, narration, or rhetorical filler.
# 8. The argument must be understandable on its own without relying on unstated assumptions.
# 9. Do not restate or reformulate the main claim in later sentences.

# Topic: {topic}
# Stance: {stance}

# Write the microtext as a single coherent paragraph.
# Do not include explanations, labels, or any additional text.
# """

def generate_microtext(topic: str, stance: str, max_new_tokens: int = 180) -> str:
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": build_user_prompt(topic, stance)},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False

    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the newly generated part (not the prompt)
    generated = out[0][inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True).strip()

    # print (messages)
    return text


SyntaxError: invalid syntax (ipython-input-2286080860.py, line 23)

In [29]:

# wiht open(original_topics)
topics = [
    "Should Germany introduce the death penalty?",
    "Should the fine for leaving dog excrements on sideways be increased?",
    "Should the morning-after pill be sold over the counter at the pharmacy?",
    "Should all universities in Germany charge tuition fees?",
    "Should shopping malls generally be allowed to open on holidays and Sundays?",
]
stances = ["PRO", "PRO", "CON"]

mictxts = []
for topic in topics:
    for stance in stances:
        text = generate_microtext(topic, stance)
        mictxts.append({"topic":topic,
                        "stance": stance,
                        "text": text})


        

# "Should all universities in Germany charge tuition fees?"
# "Should there be a cap on rent increases for a change of tenant?"
# "Should the Berlin Tegel airport remain operational after the opening of the Berlin Brandenburg airport?"

In [20]:
import json

result_dir = "/content/results"
file_name = 'final_prompt_strict.json'
os.makedirs(result_dir, exist_ok=True)
json_mictxts = json.dumps(mictxts, indent=4)
with open(os.path.join(result_dir, file_name), "w", encoding="utf-8") as json_file:
    json_file.write(json_mictxts)


In [30]:
print(mictxts)

[{'topic': 'Should all universities in Germany charge tuition fees?', 'stance': 'PRO', 'text': 'Universities in Germany should charge tuition fees to ensure financial sustainability and quality education. Tuition fees can provide essential funding for infrastructure, research, and faculty salaries, which directly enhance student learning experiences. Critics argue that charging fees may limit access for lower-income students, but this concern can be addressed through need-based scholarships and grants. Additionally, fee revenue allows institutions to invest in modern facilities and international programs, making German higher education more competitive globally. By balancing affordability with resource allocation, tuition fees support long-term institutional growth without compromising educational equity. Furthermore, introducing fees encourages greater accountability among both administrators and students.'}, {'topic': 'Should all universities in Germany charge tuition fees?', 'stance